In [2]:
import gitsource

In [1]:
import sys
print(sys.executable)

/workspaces/LLM-zoomcamp-homework/01/.venv/bin/python


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
print (documents)

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [7]:
len(documents)

72

In [7]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [8]:
question = 'How does the agentic loop keep calling the model until it stops?'

In [9]:
search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [5]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()


True

In [6]:
query ="How does the agentic loop keep calling the model until it stops?"

In [15]:


openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)
answer, usage= assistant.rag(query)
print(answer)



The agentic loop keeps calling the model by using a `while True` loop that repeats until the model returns no function calls. After each response, it checks whether any item is a `function_call`; if so, it runs the tool, appends the tool output to the message history, and loops again. If there are no function calls in that turn, it breaks out of the loop.

Filename: `01-agentic-rag/lessons/14-agentic-loop.md`


In [17]:
print(usage.input_tokens)

7146


In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
print(chunks)

[{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [9]:
len(chunks)

295

In [10]:
from minsearch import Index

index1 = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index1.fit(chunks)

In [11]:
openai_client = OpenAI()

assistant = RAGBase(
    index=index1,
    llm_client=openai_client,
)
answer1, usage1= assistant.rag(query)
print(answer1)

The loop keeps calling the model in a `while True` loop, and after each response it checks whether the model made any `function_call`s. If there were function calls, the code runs them, appends the results to `messages`, and loops again. If there are no function calls on that turn, it breaks out and stops.

File: `01-agentic-rag/lessons/14-agentic-loop.md`


In [12]:
print (usage1)

ResponseUsage(input_tokens=2329, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=91, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2420)


In [13]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [18]:
def search(query: str) -> dict[str, str]:
    """
    Search the courses  for entries matching the given query and return filename along with the answer.
    """
    return index1.search(
        query,
        num_results=5,
    )

In [19]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [20]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the courses  for entries matching the given query and return filename along with the answer.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [23]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

In [24]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [27]:
question = "How does the agentic loop work, and how is it different from plain RAG?"

In [30]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received


In [29]:
result.cost

CostInfo(input_cost=Decimal('0.00600075'), output_cost=Decimal('0.0024885'), total_cost=Decimal('0.00848925'))